# 第10回　アンサンブル学習
***
> **前提**: 第8回 Pipeline と異なり，本回は**手動前処理**で復習しながらアンサンブル学習を学びます。

## 目次
1. ランダムフォレスト
2. 勾配ブースティング
3. XGBoost
4. 特徴量重要度

---

## この回で学ぶこと

### アンサンブル学習とは

「複数の弱いモデルを組み合わせて強いモデルを作る」手法だ。選挙の多数決に例えると分かりやすい：1人の専門家より100人の多数決の方が信頼できる，という考え方と同じだ。

アンサンブル学習には大きく2つの流れがある：

```
【バギング（Bagging）】
  ランダムに異なる訓練データ → 複数の木を並列学習 → 多数決
  └ RandomForest がこれ

【ブースティング（Boosting）】
  前のモデルの間違いを次のモデルが補正 → 逐次的に学習
  └ GradientBoosting, XGBoost がこれ
```

### ランダムフォレスト（Random Forest）

- 複数の決定木をランダムなデータ・特徴量のサブセットで学習
- **過学習しにくい**：各木が異なるパターンを学習し，多数決で平均的な予測をする
- `n_estimators`：木の本数（多いほど安定するが，計算時間も増える）
- **並列化できる**ため，大規模データでも速い

### 勾配ブースティング（Gradient Boosting）

- 前の木の**残差（誤差）**を次の木が学習する逐次プロセス
- 精度は高いがチューニングが難しく，過学習しやすい
- `n_estimators`（木の本数）と `learning_rate`（学習率）のバランスが重要

### XGBoost

勾配ブースティングを大幅に改良したライブラリ：
- **2次の損失関数近似**で収束が速い
- **L1/L2 正則化**を内蔵（過学習を防ぐ）
- 欠損値を自動処理
- 並列計算・GPU 対応

Kaggle などのデータコンペで長年トップの手法として使われてきた。実務でも最も多用されるアルゴリズムの一つだ。

### 特徴量重要度（Feature Importance）の注意点

ランダムフォレストの `feature_importances_` は各特徴量が分岐にどれだけ貢献したかを示す。ただし：
- **相関する特徴量があると重要度が分散**する（連動して動く変数は各々の重要度が小さく見える）
- 重要度が高い = 目的変数の「原因」とは限らない（**相関≠因果**）
- 卒業研究で「特徴量重要度が高い変数がXXの原因だ」という解釈は慎重に行うこと

In [ ]:
%pip install -q xgboost


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

import pandas as pd

TITANIC_URL = "https://raw.githubusercontent.com/ShotaYmzk/AI-kadai/main/data/titanic/titanic.csv"

def load_titanic() -> pd.DataFrame:
    return pd.read_csv(TITANIC_URL)


## 問題1　ランダムフォレストによる生存予測
***

### `random_state` を設定する理由

`random_state=0` を指定することで，毎回同じ乱数が使われ，**結果が再現可能**になる。研究論文では再現性が必須のため，乱数シードの管理は重要なルールだ。

### `n_estimators=100` の選び方

木の本数が多いほどモデルは安定するが，計算時間が増える。一般的に：
- 100〜500: 多くの場合で十分
- 1000以上: 計算コストが高く，精度向上が頭打ちになりやすい

**実用的なアドバイス**: まず100から始め，精度に問題があれば徐々に増やす。

### 課題

タイタニックデータ（URL から読み込み）を前処理し（`Age` 欠損は中央値，`Sex` はダミー変数化），`RandomForestClassifier(n_estimators=100, random_state=0)` で生存予測モデルを学習してください。

テストデータ（20%）の正解率を出力してください。

> **比較してみよう**: 第8回の決定木（単木）の正解率と比べて，ランダムフォレストは改善されているか？複数の木を組み合わせることの効果を確認しよう。

#### Hints
- `Age` の欠損補完には `fillna` メソッドを使う。引数に列の代表値を渡す
- `Sex` のようなカテゴリ変数は数値に変換しないとモデルに渡せない。`pd.get_dummies` が便利だ
- ダミー変数化した列を元のDataFrameと結合し（`pd.concat` など）、不要になった元のカテゴリ列は削除する
- 説明変数には `Pclass`, `Age`, `SibSp`, `Parch`, `Fare` とダミー化した `Sex` 列を使う

In [ ]:
# タイタニック + RandomForest
# ここにあなたのコードを書いてください


## 問題2　勾配ブースティングの適用
***

### 乳がんデータセットについて

`load_breast_cancer()` は腫瘍の細胞核の形態学的特徴（半径，テクスチャ，周囲長など）30個から「良性/悪性」を判定する有名なベンチマークデータだ。
- サンプル数: 569件
- 特徴量: 30個（すべて数値）
- クラス: 良性（357件）/ 悪性（212件）

医療分野では「悪性を見逃さない（再現率を高める）」ことが特に重要になる（第12回で詳しく扱う）。

### 勾配ブースティングのデフォルトパラメータ

`GradientBoostingClassifier()` のデフォルト値：
- `n_estimators=100`：木の本数
- `learning_rate=0.1`：各ステップの寄与率（小さいほど慎重に学習）
- `max_depth=3`：各木の深さ（浅い木を多数組み合わせる設計）

`learning_rate` を小さくする場合，`n_estimators` を増やすと精度が維持される（トレードオフの関係）。

### 課題

乳がんデータセット `load_breast_cancer()` を用い，`GradientBoostingClassifier(random_state=0)` を学習し，テスト正解率を出力してください。

データを `train_test_split(test_size=0.2, random_state=0)` で分割してください。


In [ ]:
# GradientBoosting
# ここにあなたのコードを書いてください


## 問題3　XGBoost と3モデル比較
***

### XGBoost のインストールについて

XGBoost は scikit-learn に含まれない外部ライブラリだ。最初のセルで `%pip install xgboost` を実行している。このように機械学習の実践では様々なライブラリを組み合わせて使う。

### `eval_metric="logloss"` の意味

`logloss`（対数損失）は2値分類の損失関数で，XGBoost が内部で学習の進捗を評価するために使う。警告を抑制するために明示的に指定している。

### 3モデルの使い分けガイド

| モデル | 長所 | 短所 | 向いている場面 |
|---|---|---|---|
| RandomForest | 過学習しにくい，並列計算可能 | 精度は他より劣ることも | データが少ない，外れ値が多い |
| GradientBoosting | 精度が高い | 遅い，チューニング必要 | 精度優先，データが中程度 |
| XGBoost | 精度高い，速い，欠損値対応 | やや複雑 | コンペ，実務，大規模データ |

### 課題

同じ乳がんデータで `XGBClassifier(random_state=0, eval_metric="logloss")` を学習し，テスト正解率を出力してください。

RandomForest, GradientBoosting, XGBoost の3モデルの正解率を**比較表示**してください（print または DataFrame）。

> **考えてみよう**: このシンプルなデータでは3モデルの差は小さいかもしれない。実際の現場では，データ量・特徴量の数・欠損の有無などによってモデルの優劣が変わる。


In [ ]:
# XGBoost と3モデル比較
# ここにあなたのコードを書いてください


## 問題4　特徴量重要度の可視化と解釈
***

### 特徴量重要度とは

ランダムフォレストの各木では，特徴量を使って分岐するたびに不純度（ジニ係数やエントロピー）が減少する。`feature_importances_` は各特徴量が**全ての木での不純度減少量の合計**に占める割合を示す。合計は1になるよう正規化されている。

### グラフを重要度の降順に並べる方法

```python
indices = np.argsort(importances)[::-1]  # 降順にソート
plt.bar(range(len(names)), importances[indices])
```

降順に並べると「どの特徴量が最も影響力があるか」が一目でわかる。

### 注意：特徴量重要度の落とし穴

1. **相関特徴量がある場合**: `Age` と `Fare` が相関していると，どちらか一方の重要度が不当に低くなる
2. **カテゴリ変数のバイアス**: 取り得る値の数が多い変数ほど重要度が高く見えやすい
3. **因果関係ではない**: 「Pclass が重要」は「客室クラスが生存を決める原因だ」ではなく，「客室クラスが生存と強く関連している」という意味だ

> **卒業研究での使い方**: 特徴量重要度は「どの変数を調査すべきか」の優先度付けに使えるが，解釈は慎重に行うこと。SHAP (SHapley Additive exPlanations) というより高度な解釈手法も存在する（**発展**）。

### 課題

問題1の RandomForest モデルについて，`feature_importances_` を用いて特徴量重要度を棒グラフで可視化してください。

横軸: 特徴量名，縦軸: 重要度。**重要度の降順**に並べてください。

最も重要な特徴量は何か，その理由を考えてみてください。

#### Hints
- 学習済みモデルの `feature_importances_` 属性に各特徴量の重要度が格納されている（合計が1になる）
- 降順に並べるには、`np.argsort` で並び順のインデックスを取得し、それを使って特徴量名と重要度を同時に並べ替える
- `plt.xticks(rotation=45)` でラベルを斜めにすると読みやすくなる

In [ ]:
# 特徴量重要度の可視化
# ここにあなたのコードを書いてください
